# Privacy Meter Demo: Range Membership Inference (RaMIA)

This notebook is an interactive demonstration of running Privacy Meter to audit privacy defined by **RaMIA**. For a detailed explanation on RaMIA and how to run from bash, please refer to the [documentation](documentation/ramia.md)

## Setting up the Colab environment

If you are running it offline, you can skip to "Importing"

In [ ]:
# Clone the github repo
!git clone https://github.com/privacytrustlab/ml_privacy_meter.git

# Update the Colab environment
!pip install datasets==2.21.0 transformers==4.44.2 torch==2.4.1 torchvision==0.19.1 torchaudio

In [ ]:
# Change the directory to the cloned repo
import sys
sys.path.append('/content/ml_privacy_meter')

%cd ml_privacy_meter

/content/ml_privacy_meter


## Importing

In [1]:
import math
import time

import numpy as np
import torch
import yaml
from torch.utils.data import Subset
import matplotlib.pyplot as plt

%matplotlib inline

from audit import get_average_audit_results, audit_models_range, sample_auditing_dataset
from dataset.range_dataset import RangeDataset, RangeSampler
from get_signals import get_model_signals
from models.utils import load_models, train_models, split_dataset_for_training
from util import (
    check_configs,
    setup_log,
    initialize_seeds,
    create_directories,
    load_dataset,
)

# Enable benchmark mode in cudnn to improve performance when input sizes are consistent
torch.backends.cudnn.benchmark = True

## Load config

In [2]:
configs = "configs/ramia/cifar10.yaml"
with open(configs, "rb") as f:
        configs = yaml.load(f, Loader=yaml.Loader)

# Validate configurations
check_configs(configs)

**Using other range functions**: You can change the configuration to run Privacy Meter with different settings (e.g. range functions) by directly modifying the respective `config` fields


In [3]:
configs["ramia"]["range_function"] = "l2"

## Setting up

In [4]:
# Validate configurations
check_configs(configs)

# Initialize seeds for reproducibility
initialize_seeds(configs["run"]["random_seed"])

# Create necessary directories
log_dir = configs["run"]["log_dir"]
directories = {
    "log_dir": log_dir,
    "report_dir": f"{log_dir}/report_ramia",
    "signal_dir": f"{log_dir}/signals",
    "data_dir": configs["data"]["data_dir"],
}
create_directories(directories)

# Set up logger
logger = setup_log(
    directories["report_dir"], "time_analysis", configs["run"]["time_log"]
)

start_time = time.time()

## Load dataset

In [5]:
baseline_time = time.time()
dataset, population = load_dataset(configs, directories["data_dir"], logger)
logger.info("Loading dataset took %0.5f seconds", time.time() - baseline_time)

2026-05-06 16:31:45,631 INFO     Data loaded from data/cifar10.pkl
2026-05-06 16:31:45,639 INFO     Population data loaded from data/cifar10_population.pkl
2026-05-06 16:31:45,639 INFO     The whole dataset size: 50000
2026-05-06 16:31:45,640 INFO     Loading dataset took 0.04350 seconds


## Load or train models

In [6]:
# Define experiment parameters
num_experiments = configs["run"]["num_experiments"]
num_reference_models = configs["audit"]["num_ref_models"]
num_model_pairs = max(math.ceil(num_experiments / 2.0), num_reference_models + 1)

# Load or train models
baseline_time = time.time()
models_list, memberships = load_models(
    log_dir, dataset, num_model_pairs * 2, configs, logger
)
if models_list is None:
    # Split dataset for training two models per pair
    data_splits, memberships = split_dataset_for_training(
        len(dataset), num_model_pairs
    )
    models_list = train_models(
        log_dir, dataset, data_splits, memberships, configs, logger
    )
logger.info(
    "Model loading/training took %0.1f seconds", time.time() - baseline_time
)


2026-05-06 16:31:48,931 INFO     Loading model 0
2026-05-06 16:31:48,955 INFO     Loading model 1
2026-05-06 16:31:48,970 INFO     Loading model 2
2026-05-06 16:31:48,986 INFO     Loading model 3
2026-05-06 16:31:49,004 INFO     Model loading/training took 0.1 seconds


## Create range dataset

In [7]:
logger.info("Creating range dataset.")
dataset = RangeDataset(
    dataset,
    RangeSampler(
        range_fn=configs["ramia"]["range_function"],
        sample_size=configs["ramia"]["sample_size"],
        config=configs,
    ),
    configs,
)
logger.info("Range dataset has been created")

2026-05-06 16:31:51,203 INFO     Creating range dataset.
2026-05-06 16:31:51,204 INFO     Range dataset has been created


## Prepare auditing dataset

In [8]:
auditing_dataset, auditing_membership = sample_auditing_dataset(
        configs, dataset, logger, memberships
    )

# Also downsample the population set size if specified in the config
population = Subset(
    population,
    np.random.choice(
        len(population),
        configs["audit"].get("population_size", len(population)),
        replace=False,
    ),
)

2026-05-06 16:31:53,719 INFO     Downsampling the dataset for auditing to 10000 samples. The numbers of members and non-members are only guaranteed to be equal for the first target model, if more than one are used.


## Compute signals

In [9]:
baseline_time = time.time()
signals = get_model_signals(models_list, auditing_dataset, configs, logger)
population_signals = get_model_signals(
    models_list, population, configs, logger, is_population=True
)
logger.info("Preparing signals took %0.5f seconds", time.time() - baseline_time)

2026-05-06 16:31:56,232 INFO     Signals loaded from disk successfully.
2026-05-06 16:31:56,239 WARNING  Signals shape (10000) does not match the expected size (40000). This mismatch is likely due to a change in the training data size multiplied by ramia sample size (4).
2026-05-06 16:31:56,239 INFO     Ignoring the signals on disk and recomputing.
F:\dhoc\vande\ml_privacy_meter\venv\Lib\site-packages\torch\utils\data\dataloader.py:1118: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
2026-05-06 16:32:10,237 INFO     Computing signals for all models.
Computing softmax: 100%|█████████████████████████████████████████████████████████████████| 2/2 [00:08<00:00,  4.27s/it]
2026-05-06 16:32:44,658 INFO     Signals saved to disk.
2026-05-06 16:32:44,663 INFO     Preparing signals took 48.43665 seconds


## Computing range TPR and FPR and aggregate the auditing result

In [10]:
# Perform the privacy audit
baseline_time = time.time()
target_model_indices = list(range(num_experiments))
mia_score_list, membership_list = audit_models_range(
        f"{directories['report_dir']}/exp",
        target_model_indices,
        signals,
        population_signals,
        np.repeat(auditing_membership, configs["ramia"]["sample_size"], axis=1),
        num_reference_models,
        logger,
        configs,
    )

if len(target_model_indices) > 1:
    logger.info(
        "Auditing privacy risk took %0.1f seconds", time.time() - baseline_time
    )

# Get average audit results across all experiments
if len(target_model_indices) > 1:
    get_average_audit_results(
        directories["report_dir"], mia_score_list, membership_list, logger
    )

logger.info("Total runtime: %0.5f seconds", time.time() - start_time)

2026-05-06 16:35:23,579 INFO     Fine-tuning offline_a using paired model 1
2026-05-06 16:35:24,462 INFO     offline_a=0.00: AUC 0.5006
2026-05-06 16:35:25,290 INFO     offline_a=0.10: AUC 0.5005
2026-05-06 16:35:26,109 INFO     offline_a=0.20: AUC 0.5004
2026-05-06 16:35:26,910 INFO     offline_a=0.30: AUC 0.5003
2026-05-06 16:35:27,770 INFO     offline_a=0.40: AUC 0.5001
2026-05-06 16:35:28,587 INFO     offline_a=0.50: AUC 0.4999
2026-05-06 16:35:29,321 INFO     offline_a=0.60: AUC 0.4995
2026-05-06 16:35:30,054 INFO     offline_a=0.70: AUC 0.4989
2026-05-06 16:35:30,789 INFO     offline_a=0.80: AUC 0.4984
2026-05-06 16:35:31,524 INFO     offline_a=0.90: AUC 0.4981
2026-05-06 16:35:32,257 INFO     offline_a=1.00: AUC 0.4977
2026-05-06 16:35:32,258 INFO     The best offline_a is 0.0
2026-05-06 16:35:32,987 INFO     Finding the optimal trim ratio and direction using the paired model
2026-05-06 16:35:33,000 INFO     The optimal trim ratio is 0.50 and the direction is bottom
2026-05-06 1

<Figure size 640x480 with 0 Axes>

In [11]:
from IPython.display import Image
Image(f"{log_dir}/report_ramia/exp/ROC_0.png")

FileNotFoundError: No such file or directory: 'demo_cifar10/report_ramia/exp/ROC_0.png'

FileNotFoundError: No such file or directory: 'demo_cifar10/report_ramia/exp/ROC_0.png'

<IPython.core.display.Image object>